# Results Analysis — Lending Club (Credit-Risk: Tabular + Temporal)

**Dissertation:** Graph Neural Networks for Systemic Risk and Fraud Detection in Credit Systems
**Track:** Credit risk (tabular default prediction **+ temporal TCN**)
**Student:** Chandan Nagavolu — M.Tech, BITS Pilani WILP

Lending Club (~2.26M accepted loans) is the credit track's large, real-world
dataset and the home of the **TCN** multi-period default model (Objective 2).
Two things shape every number below:

- **Leakage control** — only origination-time features are read (an *allowlist*,
  not a drop-list), so post-origination outcome columns can never leak the label.
- **Out-of-time split** — train on the earliest vintages, test on the latest, so
  the scorecard is judged on issue periods it never saw (concept-drift realistic).

This notebook reads `reports/results_lending_club.csv`, the KS/Gini scorecard
under `reports/ks_gini/`, and `reports/lending_club_iv.csv` — all produced by
`run_lending_club_pipeline.py` (see `data/README.md`). If those files are
missing, run the pipeline first.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110


def find_project_root(start: Path) -> Path:
    """Walk up from the notebook until the folder containing CLAUDE.md."""
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "CLAUDE.md").exists():
            return cand
    raise FileNotFoundError("Could not locate project root (CLAUDE.md).")


ROOT = find_project_root(Path.cwd())
print("Project root:", ROOT)

RESULTS_CSV = ROOT / "reports" / "results_lending_club.csv"
KS_DIR = ROOT / "reports" / "ks_gini"
FIG_DIR = ROOT / "reports" / "figures"
IV_CSV = ROOT / "reports" / "lending_club_iv.csv"

MODEL_ORDER = ["logistic_regression", "random_forest", "xgboost", "tcn"]
MODEL_PRETTY = {
    "logistic_regression": "Logistic Regression", "random_forest": "Random Forest",
    "xgboost": "XGBoost", "tcn": "TCN (temporal)",
}
METRICS = ["roc_auc", "pr_auc", "ks_statistic", "gini", "f1", "mcc"]
PRETTY = {"roc_auc": "ROC-AUC", "pr_auc": "PR-AUC", "ks_statistic": "KS",
          "gini": "Gini", "f1": "F1", "mcc": "MCC"}
HAVE_RESULTS = RESULTS_CSV.exists()
print("results table present:", HAVE_RESULTS)

## 1. Why these metrics?

Lending Club defaults run ~15–20%, so the **credit-industry discrimination
metrics** are the headline:

| Metric | Why it matters here |
|---|---|
| **KS** | Max separation of the cumulative bad/good score curves. Rule of thumb: **KS > 0.40 = strong**. |
| **Gini** | `2*AUC - 1`, rank-ordering power on a 0–1 scale risk teams quote. |
| **PR-AUC** | Performance on the minority (default) class. |
| **MCC / F1** | Balanced single-number scores over the confusion matrix. |

All five models — three tabular baselines plus the **TCN** — are scored through
the same `compute_all_metrics`, so the temporal model sits in the same table.

In [ ]:
if HAVE_RESULTS:
    raw = pd.read_csv(RESULTS_CSV, parse_dates=["timestamp"]).sort_values("timestamp")
    latest = raw.groupby(["model", "split"], as_index=False).tail(1).reset_index(drop=True)
    display(latest[["model", "split", *METRICS]].round(4))
else:
    print("Run run_lending_club_pipeline.py first.")

## 2. Leaderboard — out-of-time test vintages

The headline comparison: do the deep temporal model (TCN) and the traditional
baselines separate defaulters on **future** loan vintages? KS and Gini are the
numbers a credit team would report.

In [ ]:
if HAVE_RESULTS:
    board = latest[latest.split == "test"].set_index("model").reindex(MODEL_ORDER).dropna(how="all")
    table = board[METRICS].rename(columns=PRETTY)
    table.index = [MODEL_PRETTY.get(m, m) for m in table.index]
    display(table.round(4))

In [ ]:
if HAVE_RESULTS:
    present = [m for m in MODEL_ORDER if m in board.index]
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(METRICS)); width = 0.8 / max(len(present), 1)
    for i, model in enumerate(present):
        vals = board.loc[model, METRICS].values.astype(float)
        ax.bar(x + (i - (len(present)-1)/2) * width, vals, width,
               label=MODEL_PRETTY.get(model, model), edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([PRETTY[m] for m in METRICS])
    ax.set_ylim(0, 1.05)
    ax.axhline(0.40, color="red", ls="--", lw=1, label="KS/Gini = 0.40 (strong)")
    ax.set_title("Lending Club — model metrics (out-of-time test split)")
    ax.set_ylabel("Score"); ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, axis="y", alpha=0.3); fig.tight_layout(); plt.show()

## 3. Baselines vs the temporal TCN

Does adding the temporal amortization trajectory (the TCN) beat the static
tabular baselines? **Caveat:** on the snapshot CSV the sequence is a *synthetic
amortization schedule* (proof-of-concept), so a large TCN win is not expected
until the real payment-history panel is wired in — this cell is the slot where
that comparison will live.

In [ ]:
if HAVE_RESULTS and "tcn" in board.index:
    comp = board.loc[[m for m in ["xgboost", "tcn"] if m in board.index], ["gini", "ks_statistic", "pr_auc"]]
    comp.index = [MODEL_PRETTY[m] for m in comp.index]
    display(comp.round(4))
    best_tab = board.drop(index="tcn", errors="ignore")["gini"].max()
    print(f"Best tabular Gini = {best_tab:.4f}  |  TCN Gini = {board.loc['tcn','gini']:.4f}")
else:
    print("TCN row not present yet.")

## 4. Credit scorecard — KS / Gini summary

In [ ]:
summary_path = KS_DIR / "lending_club_ks_gini_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path).round(4))
else:
    print("Run the KS/Gini scorecard step first.")

In [ ]:
# KS curves saved by the scorecard runner (one per baseline).
imgs = [FIG_DIR / f"ks_{m}_lending_club.png" for m in ["logistic_regression", "random_forest", "xgboost"]]
imgs = [p for p in imgs if p.exists()]
if imgs:
    fig, axes = plt.subplots(1, len(imgs), figsize=(6 * len(imgs), 5))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, imgs):
        ax.imshow(mpimg.imread(p)); ax.axis("off")
    fig.tight_layout(); plt.show()
else:
    print("No KS curve images found yet.")

## 5. Feature screening — Weight-of-Evidence / Information Value

IV is computed on the **train split only** (screening must not peek at
val/test). Bands: <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong,
>0.5 suspiciously strong (possible leakage — worth auditing).

In [ ]:
if IV_CSV.exists():
    iv = pd.read_csv(IV_CSV)
    display(iv.round(4))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(iv["feature"][::-1], iv["iv"][::-1], color="steelblue", edgecolor="white")
    for b in (0.02, 0.1, 0.3, 0.5):
        ax.axvline(b, color="grey", ls=":", lw=0.8)
    ax.set_xlabel("Information Value"); ax.set_title("Lending Club — IV by feature (train split)")
    ax.grid(True, axis="x", alpha=0.3); fig.tight_layout(); plt.show()
else:
    print("Run the feature step to produce the IV table.")

## 6. Takeaways

- **Leakage discipline matters most.** The allowlist + terminal-status target +
  out-of-time split are what make these KS/Gini numbers trustworthy; the usual
  inflated Lending Club results come from leaking payment-outcome columns.
- **Out-of-time is harder than random.** Reporting on future vintages is the
  realistic bar and exposes concept drift a shuffled split would hide.
- **TCN is wired but data-limited.** The temporal architecture trains and scores
  in the same harness; a genuine temporal lift needs the `PMTHIST` monthly panel
  (`sequences.mode: payment_history`) rather than the synthetic schedule.